In [1]:
%run "./common_functions"

StatementMeta(, 39d94d1a-978e-4340-8a56-a4fc2ad9cfee, 15, Finished, Available, Finished, True)

AmitWade@pwrganeshgmailcom.onmicrosoft.com


In [2]:
run_start_time = datetime.now()
# gold_yaml_path = "/lakehouse/default/Files/config/yaml/gold/"
# dummy_date = "9999-01-01 00:00:00"

StatementMeta(, 39d94d1a-978e-4340-8a56-a4fc2ad9cfee, 16, Finished, Available, Finished, False)

In [3]:
#Reading yaml files from gold
all_files = mssparkutils.fs.ls(gold_all_files)
if not all_files:
    raise FileNotFoundError(f"No YAML files found in {gold_yaml_path}")
for yaml_file in all_files:
    table_name = yaml_file.name.rsplit(".", 1)[0]
    inserted_count = 0
    updated_count = 0
    run_id = mssparkutils.env.getJobId() + f"_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    try:
        config = open_file_read_data(gold_yaml_path + yaml_file.name)["table"]

        table_name = config["name"]
        target_schema = config["schema_name"]
        source_schema = config["source_schema_name"]
        source_table_name = config.get("source_table_name", table_name)

        source_table = f"{source_schema}.{source_table_name}"
        target_table = f"{target_schema}.{table_name}"

        columns = config["columns"]
        column_names = [c["name"] for c in columns]
        column_types = {c["name"]: c["data_type"].upper() for c in columns}

        business_keys = config["business_key"]

        hash_config = config["hash_key"]
        hash_name = hash_config.get("name", "hash_key")
        excluded_columns = config["excluded_columns"]

        scd = config["scd"]
        if int(scd["type"]) != 1:
            raise ValueError(f"{table_name}: YAML must have scd.type = 1")
        inserted_col = scd["inserted_date_column"]
        updated_col = scd["updated_date_column"]

        # These columns are generated by this notebook.
        metadata_columns = [hash_name, inserted_col, updated_col]

        # All other columns are read from Silver.
        data_columns = [c for c in column_names if c not in metadata_columns]
        hash_columns = [c for c in data_columns if c not in excluded_columns]

        process_log("Start", table_name=table_name, layer="gold")

        #Read current records from Silver
        # Silver is SCD2. Only take rows marked as current.
        silver = (spark.table(source_table).filter(F.col("is_latest") == 1).select(*data_columns))

        # Cast Silver columns to the data types declared in Gold YAML.
        for c in data_columns:
            silver = silver.withColumn(c, F.col(c).cast(column_types[c]))

        #To check null and duplicate business keys.
        validate_business_keys(silver, business_keys, source_table)

        #To generate the source hash key.
        silver = generate_hash_key(silver, hash_columns, hash_name)
        silver.createOrReplaceTempView(f"tmp_{table_name}_silver")

        #Read existing Gold records and generate hash key
        gold = spark.table(target_table).select(*data_columns)
        validate_business_keys(gold, business_keys, target_table)

        # Generate the Gold hash using the same columns as Silver.
        gold = generate_hash_key(gold, hash_columns, hash_name)
        gold.createOrReplaceTempView(f"tmp_{table_name}_gold")

        # Compare Silver and Gold using runtime hash keys.
        # Both temporary views contain hash_key.
        # Neither physical Delta table contains hash_key.

        merge_join_condition = " AND ".join([f"s.{key} = g.{key}" for key in business_keys])

        # Identify new and changed records using runtime hash comparison.
        records_to_merge = spark.sql(f"""SELECT s.* FROM tmp_{table_name}_silver s LEFT JOIN tmp_{table_name}_gold g ON {merge_join_condition} WHERE g.{business_keys[0]} IS NULL    OR NOT (s.{hash_name} <=> g.{hash_name})""")

        # Remove runtime hash_key before preparing the MERGE source.
        silver_for_merge = (
            records_to_merge
            .drop(hash_name)
            .withColumn(inserted_col, F.current_timestamp())
            .withColumn(updated_col, F.current_timestamp())
            .select(*column_names)
        )

        # MERGE only new and changed records.
        if silver_for_merge.limit(1).count() > 0:
            delta_table = DeltaTable.forName(spark, target_table)

            # Match using business keys only.
            merge_condition = " AND ".join([f"t.{key} = s.{key}" for key in business_keys])

            # Update data columns, excluding business keys.
            update_values = {c: f"s.{c}" for c in data_columns if c not in business_keys}

            # Update audit columns.
            update_values[updated_col] = F.current_timestamp()

            # Insert all configured Gold columns.
            insert_values = {c: f"s.{c}" for c in column_names}
            (
                delta_table.alias("t")
                .merge(
                    silver_for_merge.alias("s"),
                    merge_condition
                )
                .whenMatchedUpdate(set=update_values)
                .whenNotMatchedInsert(values=insert_values)
                .execute()
            )

            # Get inserted and updated counts from Delta history.
            inserted_count, updated_count = get_delta_merge_counts(target_table)
        # If Silver is empty, MERGE is skipped and both counts remain zero.
        process_log("Success", table_name=table_name, layer="gold", total_inserted_count=inserted_count, total_updated_count=updated_count)
        print(f"Completed {target_table}: "f"inserted={inserted_count}, updated={updated_count}")
        # send_mail_fn( f"{pipeline_name}-{task_name}-{table_name}- Gold layer is processed successfully", f"Inserted: {inserted_count}, Updated: {updated_count}")
    except Exception as e:
        error_text = traceback.format_exc()
        print(f"Failed {table_name}: {error_text}")
        try:
            process_log("Failed", table_name=table_name, layer="gold", total_inserted_count=inserted_count, total_updated_count=updated_count, error_message=str(e))
        except Exception as log_error:
            print(f"Could not write failure log: {log_error}")
        # try:
        #     send_mail_fn( f"{pipeline_name}-{task_name}-{table_name}- Gold layer is failed", error_text)
        # except Exception as mail_error:
        #     print(f"Could not send failure email: {mail_error}")
        raise

StatementMeta(, 39d94d1a-978e-4340-8a56-a4fc2ad9cfee, 17, Finished, Available, Finished, False)

Failed customers: Traceback (most recent call last):
  File "/tmp/ipykernel_6458/3464122397.py", line 46, in <module>
    process_log("Start", table_name=table_name, layer="gold")
  File "/tmp/ipykernel_6458/632017503.py", line 34, in process_log
    pipeline_name,
    ^^^^^^^^^^^^^
NameError: name 'pipeline_name' is not defined

Could not write failure log: name 'pipeline_name' is not defined
Could not send failure email: name 'pipeline_name' is not defined


NameError: name 'pipeline_name' is not defined